In [ ]:
print('===== SPEED CLEAN: fixed bseq.dll 2-step only (loss + TONG) =====')
print('===== VERIFY LOSS FIX FINAL: bseq.dll fixed lr + prebuilt .so =====')
print('===== FIX LOSS TEST: PREBUILT .so (NO BUILD) =====')
print('===== FIX LOSS TEST: lr 0.05 -> 0.0006, 20 steps, expect loss < 10.5 =====')
# 1/4 - clone repos (fresh), dotnet
print('===== PARITY V3 (tied lm_head fixed, fresh file) =====')

# 1/4 - clone + dotnet (fault-tolerant)
# 1/4 - clone + dotnet (fault-tolerant)
print('===== ALL NATIVE KERNELS -> PURE TKV (14+7+2) + LOSS CHECK =====')
# 1/4 - clone main + prototype + Inference, install .NET (fault-tolerant)
import subprocess, os, shutil
print('===== INTEGRATE 14 PURE TKV KERNELS INTO NATIVE BUILD + LOSS CHECK =====')
for d,br in [('/content/TVmain','main'),('/content/TVcpu','prototype/cuda-backend')]:
    subprocess.run('rm -rf '+d, shell=True)
    subprocess.run('git clone --depth 1 --branch '+br+' https://github.com/nguyenhungtran18/TokenVector.git '+d, shell=True, capture_output=True, text=True, timeout=600)
    print(br, subprocess.check_output('git -C '+d+' rev-parse --short HEAD', shell=True).decode().strip())
subprocess.run('rm -rf /content/TVinf', shell=True)
subprocess.run('git clone --depth 1 https://github.com/nguyenhungtran18/TokenVector.Inference.git /content/TVinf', shell=True, capture_output=True, text=True, timeout=600)
dotnet=shutil.which('dotnet')
if not dotnet:
    print('dotnet not found -> installing .NET 10 SDK...')
    p=subprocess.run('wget -q https://dot.net/v1/dotnet-install.sh -O /tmp/dotnet-install.sh && chmod +x /tmp/dotnet-install.sh && /tmp/dotnet-install.sh --channel 10.0 --install-dir /usr/share/dotnet', shell=True, capture_output=True, text=True, timeout=900)
    print('install rc=',p.returncode)
    dotnet='/usr/share/dotnet/dotnet'
print('dotnet =',dotnet,'exists=',os.path.exists(dotnet))

In [ ]:
# 2/4 - 1 lan chay, in TRUC TIEP (khong capture_output) de khong bi "treo"
import subprocess, shutil, os, sys
shutil.copyfile('/content/TVinf/demo/tkv_simd_native_prebuilt.so','/content/TVinf/demo/tkv_simd_native.so')
cfg='/content/TVinf/tv/tests/_seq_cfg.txt'
open(cfg,'w').write('1024\n2\n1\n')
dotnet=shutil.which('dotnet') or '/usr/share/dotnet/dotnet'
env=dict(os.environ); env.pop('TKV_TORCH_GPU',None)
env['LD_LIBRARY_PATH']='/content/TVinf/demo'
print('START',flush=True)
r=subprocess.run('cd /content/TVinf && '+dotnet+' demo/bseq.dll',shell=True,
                 stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,timeout=600,env=env)
print('rc=',r.returncode,flush=True)
print((r.stdout or '')[-2500:],flush=True)
# chay them 2 lan lay median TONG (moi lan ~1s)
import re, statistics as st
tongs=[]
for k in range(2):
    q=subprocess.run('cd /content/TVinf && '+dotnet+' demo/bseq.dll',shell=True,
                     stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,timeout=600,env=env)
    m=re.search(r'TONG=([\d.]+)',q.stdout or '')
    if m: tongs.append(float(m.group(1)))
    print('run%d TONG=%s'%(k+2,m.group(1) if m else 'FAIL'),flush=True)
if tongs: print('MEDIAN TONG=%.1f us'%st.median(tongs),flush=True)

In [ ]:
# 3/4 - reference (co the bo qua)
# 3/4 - reference
# 3/4 - HF reference (same distribution weights)
# 3/4 - HF voi cung phan phoi weights: logits stats + CE
import numpy as np, torch
from transformers import GPT2Config, GPT2LMHeadModel
T=1024; V=50257; NL=12; H=768; NH=12; HD=64
f=np.float32
def gi_noise(n, seed):
    g=np.random.default_rng(seed)
    return g.standard_normal(n).astype(np.float32)*np.float32(0.02)
def ones(n): return np.ones(n,dtype=np.float32)
def zeros(n): return np.zeros(n,dtype=np.float32)
sd={}
sd['transformer.wte.weight']=gi_noise(V*H,8).reshape(V,H)
sd['transformer.wpe.weight']=gi_noise(1024*H,9).reshape(1024,H)
sd['transformer.ln_f.weight']=ones(H); sd['transformer.ln_f.bias']=zeros(H)
for li in range(NL):
    s=7+10+li*100; p='transformer.h.%d.'%li
    sd[p+'ln_1.weight']=ones(H); sd[p+'ln_1.bias']=zeros(H)
    sd[p+'attn.c_attn.weight']=gi_noise(H*3*H,s+1).reshape(H,3*H); sd[p+'attn.c_attn.bias']=zeros(3*H)
    sd[p+'attn.c_proj.weight']=gi_noise(H*H,s+2).reshape(H,H); sd[p+'attn.c_proj.bias']=zeros(H)
    sd[p+'ln_2.weight']=ones(H); sd[p+'ln_2.bias']=zeros(H)
    sd[p+'mlp.c_fc.weight']=gi_noise(H*4*H,s+3).reshape(H,4*H); sd[p+'mlp.c_fc.bias']=zeros(4*H)
    sd[p+'mlp.c_proj.weight']=gi_noise(4*H*H,s+4).reshape(4*H,H); sd[p+'mlp.c_proj.bias']=zeros(H)
sd['lm_head.weight']=sd['transformer.wte.weight']
cfg=GPT2Config(n_layer=NL,n_head=NH,n_embd=H,n_positions=1024,vocab_size=V,
               layer_norm_epsilon=1e-6,activation_function='gelu',
               resid_pdrop=0.0,attn_pdrop=0.0,embd_pdrop=0.0)
model=GPT2LMHeadModel(cfg).eval()
model.load_state_dict({k:torch.from_numpy(v.copy()) for k,v in sd.items()})
ids=np.array([(i*37+11)%V for i in range(T)],dtype=np.int64)
xi=torch.from_numpy(ids).unsqueeze(0)
with torch.no_grad():
    lg=model(input_ids=xi).logits[0].numpy()
a=lg.astype(np.float64)
print('HF DBG logits max=%.6f min=%.6f mean=%.6f'%(a.max(),a.min(),a.mean()))
print('HF DBG row0 = %.6f %.6f %.6f'%(a[0,0],a[0,1],a[0,2]))
mx=a.max(axis=1)
ce=(mx+np.log(np.exp(a-mx[:,None]).sum(axis=1))-a[np.arange(T),ids]).mean()
print('HF CE = %.6f'%ce)

In [ ]:
# 4/4 - summary
print('Compare MEDIAN TONG with baseline 28412 us (all-TKV, old lr) and 29271 us (C goc).')